In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"

In [3]:
from concept_abstraction.selection import greedy_selection_supervised, lp_selection_supervised, lp_selection_supervised_imperfect, multiple_selection_supervised, iterative_selection_supervised, greedy_selection_supervised,imperfect_lp_selection_supervised
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import torch.nn as nn
from torchvision import models
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
import pickle 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
is_jupyter = 'ipykernel' in sys.modules

In [5]:
if is_jupyter: 
    seed        = 42
    num_concepts_selected = 21
    out_folder = "cub"
    epochs=1
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--epochs', help='Number of epochs to train for',type=int, default=50)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    num_concepts_selected = args.num_concepts_selected
    out_folder = args.out_folder
    epochs = args.epochs 

save_name = secrets.token_hex(4)  

In [6]:
results = {}
results['parameters'] = {'seed'      : seed,
        'num_concepts_selected': num_concepts_selected,
        'epochs': epochs,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'num_concepts_selected': 21, 'epochs': 1}


In [7]:
np.random.seed(seed)
random.seed(seed)

In [8]:
def get_performance(selected_concepts,accuracy_by_concept):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

## Perfrect Concepts

In [9]:
results['perfect'] = {}

In [10]:
train = pickle.load(open("../../data/cub/train.pkl","rb"))
test = pickle.load(open("../../data/cub/test.pkl","rb"))

In [11]:
train_X = np.array([i['attribute_label'] for i in train])
train_Y = np.array([i['class_label'] for i in train])
test_X = np.array([i['attribute_label'] for i in test])
test_Y = np.array([i['class_label'] for i in test])

In [12]:
results['perfect']['lp'] = {}
for c in range(20,num_concepts_selected,20):
    lp_concept_list = lp_selection_supervised(train_X,train_Y,c)
    results['perfect']['lp'][c] = {
        'reward': get_performance(lp_concept_list,np.ones(312)),
        'concepts': lp_concept_list
    }
print("Finished LP")

Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17

Interrupt request received
Finished LP


In [ ]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]
results['perfect']['manual'] = {
    'reward': get_performance(manually_selected_concepts,np.ones(312)),
    'concepts': manually_selected_concepts
}
print("Manual Performance {}".format(results['perfect']['manual']['reward']))

Manual Performance 1.0


#### Imperfect Concepts

In [16]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

train = pickle.load(open("../../data/cub/train_error.pkl","rb"))
test = pickle.load(open("../../data/cub/test_error.pkl","rb"))

pred_train_X = sigmoid(np.array([i['attribute_label'] for i in train])).round()
train_Y = np.array([i['class_label'] for i in train])
pred_test_X = sigmoid(np.array([i['attribute_label'] for i in test])).round()
test_Y = np.array([i['class_label'] for i in test])

In [17]:
from sklearn.neural_network import MLPClassifier

def get_performance_real(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(pred_train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(pred_test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

In [18]:
results['imperfect'] = {}

In [19]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]


In [31]:
results['imperfect']['manual'] = {'reward': get_performance_real(manually_selected_concepts), 'concepts': manually_selected_concepts}

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [32]:
results['imperfect']['lp'] = {}
for c in results['perfect']['lp']:
    results['imperfect']['lp'][c] = {
        'reward': get_performance_real(results['perfect']['lp'][c]['concepts']),
        'concepts': c
    }

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [33]:
results['imperfect']['multiple'] = {}
for c in results['perfect']['lp']:
    imperfect_concepts = multiple_selection_supervised(train_X,train_Y,c)
    results['imperfect']['multiple'][c] = {
        'reward': get_performance_real(imperfect_concepts), 
        'concepts': imperfect_concepts
    }
results['imperfect']['multiple']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.5338280980324474,
  'concepts': [6,
   20,
   35,
   51,
   54,
   90,
   117,
   132,
   149,
   151,
   163,
   178,
   209,
   212,
   218,
   235,
   236,
   240,
   289,
   304]}}

In [34]:
results['imperfect']['iterative'] = {}
all_imperfect_concepts = iterative_selection_supervised(pred_train_X,train_Y,num_concepts_selected)
all_imperfect_concepts = [int(i) for i in all_imperfect_concepts]

for c in results['perfect']['lp']:
    imperfect_concepts = all_imperfect_concepts[:c]
    results['imperfect']['iterative'][c] = {
        'reward': get_performance_real(imperfect_concepts), 
        'concepts': imperfect_concepts
    }
results['imperfect']['iterative']

On iteration 0
On iteration 1


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.38125647221263376,
  'concepts': [6,
   20,
   35,
   51,
   209,
   212,
   235,
   236,
   240,
   289,
   3,
   11,
   22,
   26,
   34,
   37,
   41,
   46,
   49,
   52]}}

In [35]:
results['imperfect']['random'] = {}

for c in results['perfect']['lp']:
    random_concepts = random.sample(list(range(312)),c)
    results['imperfect']['random'][c] = {
        'reward': get_performance_real(random_concepts), 
        'concepts': random_concepts
    }
results['imperfect']['random']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.11529168104936141,
  'concepts': [57,
   12,
   140,
   125,
   114,
   71,
   52,
   279,
   44,
   302,
   216,
   16,
   15,
   47,
   111,
   119,
   258,
   308,
   13,
   287]}}

In [36]:
results['imperfect']['greedy'] = {}

for c in results['perfect']['lp']:
    greedy_concepts = greedy_selection_supervised(train_X,train_Y,c)
    results['imperfect']['greedy'][c] = {
        'reward': get_performance_real(greedy_concepts), 
        'concepts': greedy_concepts
    }
results['imperfect']['greedy']

[0.24998083 0.24995266 0.24560361 0.24473498 0.24395134 0.24265612
 0.24214729 0.241924   0.2411571  0.2411571  0.23769547 0.2339816
 0.23339578 0.2330718  0.23246971 0.22802237 0.22429046 0.22422355
 0.2238877  0.21352187] 0.0543086312740497


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.5329651363479462,
  'concepts': [236,
   289,
   240,
   51,
   212,
   20,
   35,
   209,
   235,
   6,
   178,
   218,
   132,
   117,
   151,
   54,
   163,
   149,
   304,
   90]}}

## Intervention

In [13]:
test = pickle.load(open("../../data/cub/test_error.pkl","rb"))


In [14]:
cub_s = json.load(open("../../data/cub/cub_s.json"))
full_test = pickle.load(open("../../data/cub/test.pkl","rb"))

In [20]:
accuracy = []
machine_accuracy = []

for i in cub_s:
    for j in cub_s[i]:
        corresponding_row = [k['attribute_label'] for k in full_test if k['id'] == int(i)][0]
        corresponding_preds = sigmoid(np.array([k['attribute_label'] for k in test if k['id'] == int(i)][0]))

        accuracy.append(np.abs(np.array(corresponding_row)-np.array(j)))
        machine_accuracy.append(np.abs(np.array(corresponding_row)-np.array(corresponding_preds)))
human_accuracy = 1-np.mean(accuracy,axis=0)
machine_accuracy = 1-np.mean(machine_accuracy,axis=0)

In [23]:
s = [i for i in range(len(human_accuracy)) if human_accuracy[i]>machine_accuracy[i]]

In [25]:
[i for i in s if i in manually_selected_concepts]

[203]

In [21]:
manually_selected_concepts

[1,
 4,
 6,
 7,
 10,
 14,
 15,
 20,
 21,
 23,
 25,
 29,
 30,
 35,
 36,
 38,
 40,
 44,
 45,
 50,
 51,
 53,
 54,
 56,
 57,
 59,
 63,
 64,
 69,
 70,
 72,
 75,
 80,
 84,
 90,
 91,
 93,
 99,
 101,
 106,
 110,
 111,
 116,
 117,
 119,
 125,
 126,
 131,
 132,
 134,
 145,
 149,
 151,
 152,
 153,
 157,
 158,
 163,
 164,
 168,
 172,
 178,
 179,
 181,
 183,
 187,
 188,
 193,
 194,
 196,
 198,
 202,
 203,
 208,
 209,
 211,
 212,
 213,
 218,
 220,
 221,
 225,
 235,
 236,
 238,
 239,
 240,
 242,
 243,
 244,
 249,
 253,
 254,
 259,
 260,
 262,
 268,
 274,
 277,
 283,
 289,
 292,
 293,
 294,
 298,
 299,
 304,
 305,
 308,
 309,
 310,
 311]

In [37]:
np.mean(human_accuracy)

0.8946838087006626

In [72]:
num_concepts = len(manually_selected_concepts)
lp_selection  = lp_selection_supervised(train_X,train_Y,num_concepts)
multiple_selection  = multiple_selection_supervised(train_X,train_Y,num_concepts)
iterative_selection = iterative_selection_supervised(pred_train_X,train_Y,num_concepts)
greedy_selection = greedy_selection_supervised(train_X,train_Y,num_concepts)
manual_selection = manually_selected_concepts
random_selection = random.sample(list(range(312)),num_concepts)
accuracies = np.mean(test_X == pred_test_X,axis=0)
imperfect_selection = imperfect_lp_selection_supervised(train_X,train_Y,num_concepts,accuracies)

On iteration 0
On iteration 1
On iteration 2
On iteration 3
On iteration 4
On iteration 5
On iteration 6
On iteration 7
On iteration 8
On iteration 9
On iteration 10
[0.24998083 0.24995266 0.24560361 0.24473498 0.24395134 0.24265612
 0.24214729 0.241924   0.2411571  0.2411571  0.23769547 0.2339816
 0.23339578 0.2330718  0.23246971 0.22802237 0.22429046 0.22422355
 0.2238877  0.21352187 0.20414544 0.2031579  0.19966545 0.19853649
 0.19691569 0.19340045 0.18822764 0.18770833 0.18486649 0.18390512
 0.17921471 0.17686562 0.17527825 0.1747072  0.17390409 0.17111819
 0.17029621 0.16672516 0.16282978 0.16270661 0.15972456 0.15922268
 0.15244146 0.15205032 0.15152757 0.14568555 0.14487586 0.13981365
 0.13702776 0.13660688 0.1354807  0.12947695 0.12787919 0.12612418
 0.12612418 0.12597737 0.12597737 0.12287415 0.1224277  0.12168189
 0.11837447 0.11209834 0.11147821 0.1110122  0.1110122  0.10835669
 0.10835669 0.10757088 0.10646708 0.10615093 0.10504166 0.10152765
 0.10120611 0.10056198 0.100400

In [73]:
n_rows, n_cols = test_X.shape

# Generate random mask of same shape as test_X
random_vals = np.random.rand(n_rows, n_cols)

# Make a copy
human_test_X = test_X.copy()

# For each column, flip values with probability (1 - human_accuracy[i])
for i in range(n_cols):
    flip_mask = random_vals[:, i] > human_accuracy[i]
    human_test_X[flip_mask, i] = 1 - human_test_X[flip_mask, i]


In [87]:
def get_performance_real(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(pred_train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(intervention_test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

In [75]:
results['intervention'] = {}

In [90]:
for intervention_percent in [0.2,0.4,0.6,0.8,1.0]:
    mask = np.random.rand(*test_X.shape) < intervention_percent

    # Build the mixed array
    intervention_test_X = np.where(mask, test_X, pred_test_X)

    for arr,description in zip([manually_selected_concepts,
                                lp_selection,
                                multiple_selection,
                                iterative_selection,
                                greedy_selection,
                                random_selection,
                                imperfect_selection],[
                                    "manual","lp","multiple",
                                    'iterative','greedy','random',
                                    'imperfect'
                                ]):
        if description not in results['intervention']:
            results['intervention'][description] = {}
        results['intervention'][description][intervention_percent] = {
            'reward': get_performance_real(arr)
        }
        print(description,intervention_percent,results['intervention'][description][intervention_percent]['reward'])


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


manual 0.2 0.6717293752157404


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


lp 0.2 0.6475664480497066


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


...

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


iterative 1.0 0.9656541249568519


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


greedy 1.0 0.9948222298929927
random 1.0 0.9487400759406283


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [79]:
results['intervention_human'] = {}

In [81]:
for intervention_percent in [0.2,0.4,0.6,0.8,1.0]:
    mask = np.random.rand(*human_test_X.shape) < intervention_percent

    # Build the mixed array
    intervention_test_X = np.where(mask, human_test_X, pred_test_X)

    for arr,description in zip([manually_selected_concepts,
                                lp_selection,
                                multiple_selection,
                                iterative_selection,
                                greedy_selection,
                                random_selection,
                                imperfect_selection],[
                                    "manual","lp","multiple",
                                    'iterative','greedy','random',
                                    'imperfect'
                                ]):
        if description not in results['intervention_human']:
            results['intervention_human'][description] = {}
        results['intervention_human'][description][intervention_percent] = {
            'reward': get_performance_real(arr)
        }
        print(description,intervention_percent,results['intervention_human'][description][intervention_percent]['reward'])


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


manual 0.2 0.6146013117017605


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


lp 0.2 0.5969968933379358


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


...

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


multiple 1.0 0.4111149464963756
iterative 1.0 0.3432861580945806


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


greedy 1.0 0.38073869520193304
random 1.0 0.3066965826717294
imperfect 1.0 0.24646185709354504


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


## Save Data

In [38]:
save_path = get_save_path(out_folder,save_name)

In [39]:
delete_duplicate_results(out_folder,"",results)

An error occurred: Expected object or value
An error occurred: Expected object or value
An error occurred: Expected object or value
An error occurred: Expected object or value


In [40]:
json.dump(results,open('../../results/'+save_path,'w'))